In [ ]:
import pandas as pd
import numpy as np

## Import dei dati
Acquisire i dati presenti nei vari fogli del file Excel Database_compito_Pandas.xlsx

In [ ]:
dfs = pd.read_excel("Database_compito_Pandas.xlsx",sheet_name=None)

## Esercizio 1

Scrivi una query per estrarre le fatture che rispettano
almeno una tra queste condizioni
- data fattura compresa tra 2 novembre 2018 e 1 gennaio 2019 (estremi eslusi)
- spedizione non valorizzata
- id cliente 1 o 3.

In particolare visualizzare in output 
- l'IdFattura
- una colonna di nome Esito Consegna valorizzata con "In orario" se la DataArrivoEffettiva
è minore o uguale alla DataArrivoRichiesta, "In ritardo" altrimenti
(altrimenti NULL)

In [ ]:
es_1 = dfs["Fatture"].query(" (DataFattura > '2018-11-02' and DataFattura < '2019-01-01') \
                      or Spedizione.isna() \
                      or IdCliente in (1,3) ")[["IdFattura","DataArrivoEffettiva","DataArrivoRichiesta"]].copy()

In [ ]:
es_1["EsitoConsegne"]=np.where(es_1["DataArrivoEffettiva"]>es_1["DataArrivoRichiesta"],"In orario", "In ritardo")

In [ ]:
es_1 = es_1.drop(["DataArrivoEffettiva","DataArrivoRichiesta"],axis=1)

In [ ]:
es_1

## Esercizio 2:

Scrivi una query per estrarre l'elenco delle Denominazioni dei fornitori
che sono presenti in almeno una fattura nel periodo tra il primo agosto 2018 e il 
15 agosto 2018 (considerare la DataFattura, estremi inclusi)


In [ ]:
es_2 = dfs["Fatture"].query(" DataFattura >= '2018-08-01' and DataFattura <= '2018-08-15' ")[["IdFornitore"]].drop_duplicates().copy()

In [ ]:
es_2 = pd.merge(es_2,dfs["Fornitori"],on="IdFornitore",how="inner")[["Denominazione"]]

In [ ]:
es_2

## Esercizio 3:

Scrivi una query per estrarre per ogni anno
il numero di clienti a cui è stata emessa almeno una fattura in quell'anno.
(Considerare la DataFattura)

In [ ]:
dfs["Fatture"]["Anno"]=dfs["Fatture"]["DataFattura"].dt.year

In [ ]:
es3 = dfs["Fatture"].query("IdCliente.notna()")[["Anno","IdCliente"]].drop_duplicates().copy()

In [ ]:
es3 = es3.groupby(by = "Anno",
                  as_index=False, 
                  dropna=False)\
         .agg(NumeroClienti = ("IdCliente",np.size))

In [ ]:
es3

Altro metodo che scarta anche i null senza dover filtrare ( di default DropNa=True)

In [ ]:
dfs["Fatture"].groupby(by = "Anno",
                  as_index=False, 
                  dropna=False)\
                 ["IdCliente"].nunique()\
                 .rename(columns={"IdCliente":"NumeroClienti"})

## Esercizio 4
Scrivi la query per estrarre la combinazione di anno e mese dove 
si è registrato il numero più alto di clienti che hanno fatto almeno una fattura in quell'anno-mese.
Gestire eventuali pari-merito a piacere.
(Considerare la DataFattura)

In [ ]:
dfs["Fatture"]["Anno"]=dfs["Fatture"]["DataFattura"].dt.year
dfs["Fatture"]["Mese"]=dfs["Fatture"]["DataFattura"].dt.month

In [ ]:
es4 = dfs["Fatture"].query("IdCliente.notna()")[["Anno","Mese","IdCliente"]].drop_duplicates().copy()

In [ ]:
es4 = es4.groupby(by = ["Anno","Mese"],
                  as_index=False, 
                  dropna=False)\
         .agg(NumeroClienti = ("IdCliente",np.size))

In [ ]:
es4 = es4.sort_values(by="NumeroClienti", ascending=False).head(1)

In [ ]:
es4

In [ ]:
#oppure
dfs["Fatture"].groupby(by = ["Anno","Mese"],
                       as_index=False, 
                        dropna=False)["IdCliente"]\
              .nunique()\
              .sort_values(by="IdCliente", ascending=False)\
              .head(1)\
              .rename(columns={"IdCliente":"NumeroClienti"})